## 3. Generate files first 

card_faces, cards, regions, registration, rules, scoring, synthetic_cannon, tracking - these .py files need to be generated first for running the other cells. If they are present already, nothing needs to be done

Copied verbatim from the repo (unit-tested there); regenerate with `tools/build_colab_notebook.py`.


### Procedural deck and the canonical-space scene compositor

In [9]:
%%writefile cards.py
"""Core card model for the game *29*.

The deck is a fixed closed set of 32 cards: 8 ranks x 4 suits.

Two orderings matter and they are DIFFERENT:
  * point value   -- how many points the card is worth when captured
  * trick strength -- which card beats which within a suit

Standard 29:
    points:   J=3, 9=2, A=1, 10=1, K=0, Q=0, 8=0, 7=0   (28 total)
    strength (high->low): J, 9, A, 10, K, Q, 8, 7
"""

from __future__ import annotations

from dataclasses import dataclass
from enum import Enum


class Suit(Enum):
    SPADES = "S"
    HEARTS = "H"
    DIAMONDS = "D"
    CLUBS = "C"

    @property
    def symbol(self) -> str:
        return {"S": "♠", "H": "♥", "D": "♦", "C": "♣"}[self.value]


class Rank(Enum):
    SEVEN = "7"
    EIGHT = "8"
    NINE = "9"
    TEN = "10"
    JACK = "J"
    QUEEN = "Q"
    KING = "K"
    ACE = "A"


# Point value captured when a card is won in a trick. Sums to 28 across the deck.
RANK_POINTS: dict[Rank, int] = {
    Rank.JACK: 3,
    Rank.NINE: 2,
    Rank.ACE: 1,
    Rank.TEN: 1,
    Rank.KING: 0,
    Rank.QUEEN: 0,
    Rank.EIGHT: 0,
    Rank.SEVEN: 0,
}

# Trick-taking strength, high -> low. Larger int beats smaller (within a suit).
_RANK_STRENGTH_ORDER: list[Rank] = [
    Rank.JACK,
    Rank.NINE,
    Rank.ACE,
    Rank.TEN,
    Rank.KING,
    Rank.QUEEN,
    Rank.EIGHT,
    Rank.SEVEN,
]
# Assign so that JACK is highest.
RANK_STRENGTH: dict[Rank, int] = {
    rank: len(_RANK_STRENGTH_ORDER) - 1 - i
    for i, rank in enumerate(_RANK_STRENGTH_ORDER)
}


@dataclass(frozen=True)
class Card:
    """An immutable, hashable card. Suitable as a dict key / set member."""

    rank: Rank
    suit: Suit

    @property
    def points(self) -> int:
        return RANK_POINTS[self.rank]

    @property
    def strength(self) -> int:
        """Rank strength for trick comparison (suit-agnostic; compare within a suit)."""
        return RANK_STRENGTH[self.rank]

    @property
    def code(self) -> str:
        """Compact code like ``'JS'`` (Jack of Spades) or ``'10H'``."""
        return f"{self.rank.value}{self.suit.value}"

    def __str__(self) -> str:  # pragma: no cover - display only
        return f"{self.rank.value}{self.suit.symbol}"

    @classmethod
    def from_code(cls, code: str) -> "Card":
        """Parse ``'JS'`` / ``'10H'`` / ``'7c'`` back into a Card (case-insensitive)."""
        code = code.strip().upper()
        suit_char = code[-1]
        rank_str = code[:-1]
        try:
            suit = Suit(suit_char)
        except ValueError as exc:
            raise ValueError(f"Unknown suit in card code {code!r}") from exc
        try:
            rank = Rank(rank_str)
        except ValueError as exc:
            raise ValueError(f"Unknown rank in card code {code!r}") from exc
        return cls(rank, suit)


def full_deck() -> list[Card]:
    """Return all 32 distinct cards of the fixed *29* deck."""
    return [Card(rank, suit) for suit in Suit for rank in Rank]


# Convenience: total points in the deck (invariant == 28).
DECK_TOTAL_POINTS = sum(c.points for c in full_deck())

Overwriting cards.py


In [10]:
%%writefile rules.py
"""Trick-taking rules for *29*: play model, trick-winner, and marriage detection.

These functions are the deterministic heart of the system. They take *already
identified* cards (rank+suit + which player played them) and produce trick
winners, captured points, and marriage events. They contain no CV logic, so they
can be unit-tested exhaustively against hand-computed games.

Player / team model
-------------------
Players are labelled 1..4 by seat. Players sitting opposite are partners:
    Team A = {1, 3}
    Team B = {2, 4}
"""

from __future__ import annotations

from dataclasses import dataclass

from .cards import Card, Rank, Suit

PLAYERS = (1, 2, 3, 4)
TEAM_A = frozenset({1, 3})
TEAM_B = frozenset({2, 4})

# Seating order around the table. Partners sit opposite, so going round the table
# alternates teams: 1 (A), 2 (B), 3 (A), 4 (B).
SEATING_ORDER = (1, 2, 3, 4)


def order_from_leader(leader: int) -> list[int]:
    """Return the play order for a trick led by ``leader``, going round the table.

    Play order is not observed from the video -- it is derived. The winner of a
    trick leads the next one, so once the first leader is known every subsequent
    order follows from the trick outcomes.
    """
    if leader not in SEATING_ORDER:
        raise ValueError(f"leader must be one of {SEATING_ORDER}, got {leader!r}")
    start = SEATING_ORDER.index(leader)
    return [SEATING_ORDER[(start + step) % len(SEATING_ORDER)] for step in range(4)]


def team_of(player: int) -> frozenset[int]:
    """Return the partner-set (team) that ``player`` belongs to."""
    if player in TEAM_A:
        return TEAM_A
    if player in TEAM_B:
        return TEAM_B
    raise ValueError(f"player must be one of {PLAYERS}, got {player!r}")


@dataclass(frozen=True)
class Play:
    """A single card played by a player. ``order`` is 0..3 within the trick."""

    player: int
    card: Card
    order: int


@dataclass(frozen=True)
class Trick:
    """The four plays of one trick, in play order (index 0 == leader)."""

    plays: tuple[Play, ...]

    def __post_init__(self) -> None:
        if len(self.plays) != 4:
            raise ValueError(f"a trick has exactly 4 plays, got {len(self.plays)}")
        players = {p.player for p in self.plays}
        if players != set(PLAYERS):
            raise ValueError(f"a trick must have all of players {PLAYERS}, got {players}")
        cards = [p.card for p in self.plays]
        if len(set(cards)) != 4:
            raise ValueError(f"duplicate card within a trick: {cards}")

    @property
    def leader(self) -> int:
        return self.ordered()[0].player

    @property
    def led_suit(self) -> Suit:
        return self.ordered()[0].card.suit

    def ordered(self) -> list[Play]:
        """Plays sorted by play order (leader first)."""
        return sorted(self.plays, key=lambda p: p.order)

    @property
    def points(self) -> int:
        """Total captured points available in this trick."""
        return sum(p.card.points for p in self.plays)

    @classmethod
    def from_mapping(
        cls, player_to_card: dict[int, Card], order: list[int] | None = None
    ) -> "Trick":
        """Build a Trick from ``{player: card}`` plus an optional play order.

        ``order`` is the list of players in the sequence they played
        (e.g. ``[2, 3, 4, 1]``). Defaults to ``[1, 2, 3, 4]``.
        """
        order = order or list(PLAYERS)
        if sorted(order) != list(PLAYERS):
            raise ValueError(f"order must be a permutation of {PLAYERS}, got {order}")
        plays = tuple(
            Play(player=pl, card=player_to_card[pl], order=i)
            for i, pl in enumerate(order)
        )
        return cls(plays)


def trick_winner(trick: Trick, trump_suit: Suit | None) -> int:
    """Return the player who wins ``trick``.

    Rules:
      * If any trump was played, the highest-strength trump wins.
      * Otherwise the highest-strength card of the *led* suit wins.
      * ``trump_suit=None`` means trump is not (yet) active -> led-suit only.

    Note on reveal timing: trump only becomes active once revealed. Callers that
    model a mid-game reveal should pass ``trump_suit=None`` for tricks played
    before trump is active and the actual suit from the reveal trick onward.
    """
    ordered = trick.ordered()
    if trump_suit is not None:
        trumps = [p for p in ordered if p.card.suit == trump_suit]
        if trumps:
            return max(trumps, key=lambda p: p.card.strength).player

    led = trick.led_suit
    followers = [p for p in ordered if p.card.suit == led]
    # followers is always non-empty (the leader itself follows the led suit).
    return max(followers, key=lambda p: p.card.strength).player


def detect_marriage(
    tricks: list[Trick],
    trump_suit: Suit | None,
    reveal_trick_index: int | None,
) -> int | None:
    """Return the player who scored a marriage, or ``None``.

    Marriage (per project spec): the King AND Queen of the *trump* suit are both
    played by the *same* player, *after* the trump reveal.

    ``reveal_trick_index`` is the index into ``tricks`` at which trump becomes
    active (inclusive). Plays in earlier tricks do not count toward marriage.
    ``None`` for either trump argument means no marriage is possible.
    """
    if trump_suit is None or reveal_trick_index is None:
        return None

    king = Card(Rank.KING, trump_suit)
    queen = Card(Rank.QUEEN, trump_suit)
    king_by: int | None = None
    queen_by: int | None = None

    for idx, trick in enumerate(tricks):
        if idx < reveal_trick_index:
            continue
        for play in trick.plays:
            if play.card == king:
                king_by = play.player
            elif play.card == queen:
                queen_by = play.player

    if king_by is not None and king_by == queen_by:
        return king_by
    return None

Overwriting rules.py


In [11]:
%%writefile scoring.py
"""Hand-level scoring for *29*: tally team points and resolve the bid.

Consumes the deterministic outputs of :mod:`rules` (trick winners, captured
points, marriage) and produces the final result of a hand: each team's score and
whether the bidding team made its contract.
"""

from __future__ import annotations

from dataclasses import dataclass, field

from .cards import Card, Suit
from .rules import (
    TEAM_A,
    TEAM_B,
    Trick,
    detect_marriage,
    order_from_leader,
    team_of,
    trick_winner,
)

# Marriage bonus points. Configurable; 4 is the common value in *29*.
DEFAULT_MARRIAGE_BONUS = 4


def _team_name(team: frozenset[int]) -> str:
    return "A" if team == TEAM_A else "B"


@dataclass
class TrumpInfo:
    """Describes the trump for a hand.

    * ``suit``               -- the trump suit, or None if never revealed on camera.
    * ``reveal_trick_index`` -- index into the hand's trick list at which trump
      becomes active (inclusive). None if trump was never revealed.
    """

    suit: Suit | None = None
    reveal_trick_index: int | None = None

    @property
    def is_active(self) -> bool:
        return self.suit is not None and self.reveal_trick_index is not None


@dataclass
class HandResult:
    team_points: dict[str, int]
    trick_winners: list[int]
    marriage_player: int | None
    marriage_team: str | None
    # Leader of each trick, in order. Derived from the previous trick's winner,
    # never observed from the video.
    trick_leaders: list[int] = field(default_factory=list)
    bidding_team: str | None = None
    bid: int | None = None
    contract_made: bool | None = None
    winning_team: str | None = None
    detail: list[dict] = field(default_factory=list)


def score_hand(
    tricks: list[Trick],
    trump: TrumpInfo,
    *,
    bidding_team: str | None = None,
    bid: int | None = None,
    marriage_bonus: int = DEFAULT_MARRIAGE_BONUS,
) -> HandResult:
    """Score one hand (one deal of 8 tricks) and resolve the bid if given.

    ``bidding_team`` is ``"A"`` or ``"B"`` (from user text input); ``bid`` is the
    contracted point target. If both are supplied, the bidding team must capture
    >= ``bid`` points (including any marriage bonus it earned) to make contract;
    otherwise the opposing team wins the hand.
    """
    points = {"A": 0, "B": 0}
    winners: list[int] = []
    detail: list[dict] = []

    for idx, trick in enumerate(tricks):
        # Trump is only active from the reveal trick onward.
        active_suit = (
            trump.suit
            if trump.is_active and idx >= trump.reveal_trick_index  # type: ignore[operator]
            else None
        )
        winner = trick_winner(trick, active_suit)
        winners.append(winner)
        team = _team_name(team_of(winner))
        points[team] += trick.points
        detail.append(
            {
                "trick": idx,
                "winner": winner,
                "team": team,
                "points": trick.points,
                "trump_active": active_suit.value if active_suit else None,
            }
        )

    # Marriage bonus.
    marriage_player = detect_marriage(tricks, trump.suit, trump.reveal_trick_index)
    marriage_team: str | None = None
    if marriage_player is not None:
        marriage_team = _team_name(team_of(marriage_player))
        points[marriage_team] += marriage_bonus

    result = HandResult(
        team_points=points,
        trick_winners=winners,
        marriage_player=marriage_player,
        marriage_team=marriage_team,
        bidding_team=bidding_team,
        bid=bid,
        detail=detail,
    )

    result.trick_leaders = [t.leader for t in tricks]

    # Resolve the contract.
    if bidding_team is not None and bid is not None:
        if bidding_team not in ("A", "B"):
            raise ValueError(f"bidding_team must be 'A' or 'B', got {bidding_team!r}")
        made = points[bidding_team] >= bid
        result.contract_made = made
        other = "B" if bidding_team == "A" else "A"
        result.winning_team = bidding_team if made else other

    return result


def build_tricks_from_plays(
    plays: list[dict[int, Card]],
    trump: TrumpInfo,
    first_leader: int,
) -> list[Trick]:
    """Turn per-trick ``{player: card}`` mappings into ordered :class:`Trick` objects.

    Vision recovers *who played what*, not the sequence. The sequence is derived:
    the winner of a trick leads the next, going round the table from there. So the
    whole chain follows from ``first_leader``.

    This is not bookkeeping -- the leader fixes the *led suit*, which decides the
    winner whenever no trump is played. Get a leader wrong and the winner can
    change, which changes the next leader, so an early error propagates.
    """
    tricks: list[Trick] = []
    leader = first_leader

    for index, mapping in enumerate(plays):
        trick = Trick.from_mapping(mapping, order=order_from_leader(leader))
        tricks.append(trick)

        active_suit = (
            trump.suit
            if trump.is_active and index >= trump.reveal_trick_index  # type: ignore[operator]
            else None
        )
        leader = trick_winner(trick, active_suit)

    return tricks


def score_hand_from_plays(
    plays: list[dict[int, Card]],
    trump: TrumpInfo,
    *,
    first_leader: int = 1,
    bidding_team: str | None = None,
    bid: int | None = None,
    marriage_bonus: int = DEFAULT_MARRIAGE_BONUS,
) -> HandResult:
    """Score a hand straight from ``{player: card}`` mappings, deriving play order.

    ``first_leader`` is the only ordering input the video cannot supply; in *29* it
    is the bid winner. Every later leader is computed from the trick outcomes.
    """
    tricks = build_tricks_from_plays(plays, trump, first_leader)
    return score_hand(
        tricks,
        trump,
        bidding_team=bidding_team,
        bid=bid,
        marriage_bonus=marriage_bonus,
    )

Overwriting scoring.py


In [12]:
%%writefile registration.py
"""Table registration: map a drifting hand-held view onto a canonical top-down table.

The gameplay footage is shot hand-held at an oblique angle, so the table wanders
around the frame between (and within) tricks. Every downstream notion of "where a
card is" -- which arm of the trick cross, the trump spot, the counter spot -- is
only meaningful in a frame of reference attached to the *table*, not to the image.

The round table projects to an ellipse. We fit that ellipse on the cane (rattan)
surface, then apply the affine map that turns the ellipse back into a circle of
radius ``CANON_RADIUS`` centred in a ``CANON_SIZE`` square. That removes the
translation, scale and foreshortening drift, which is what actually moves between
frames. It deliberately stops short of a full perspective rectification: recovering
a true homography from a single conic is ambiguous without a second cue, and the
affine approximation is both stable and sufficient for region assignment.

In-plane orientation is preserved (the ellipse's tilt is re-applied after the
circularising scale) so that "players sit along the top edge" stays true in
canonical space.
"""

from __future__ import annotations

from dataclasses import dataclass

import cv2
import numpy as np

# Canonical table space: a CANON_SIZE x CANON_SIZE image with the table drawn as a
# centred circle of radius CANON_RADIUS.
CANON_SIZE = 512
CANON_RADIUS = 230.0

# Cane-surface colour gate in HSV. The table top is a warm tan/orange weave; the
# floor tiles around it are desaturated pink-grey and the rim is near-black.
_CANE_LOWER = np.array([5, 60, 80], dtype=np.uint8)
_CANE_UPPER = np.array([35, 255, 255], dtype=np.uint8)

# Plausibility gates on a candidate ellipse, as fractions of frame area / extent.
_MIN_AREA_FRAC = 0.04
_MAX_AREA_FRAC = 0.90
_MAX_AXIS_RATIO = 4.0


@dataclass(frozen=True)
class TableFit:
    """A fitted table ellipse plus the affine map into canonical table space."""

    cx: float
    cy: float
    major: float  # full length of the major axis, pixels
    minor: float  # full length of the minor axis, pixels
    angle: float  # major-axis tilt in degrees, OpenCV convention
    score: float  # fraction of the ellipse actually covered by cane pixels

    @property
    def center(self) -> tuple[float, float]:
        return self.cx, self.cy

    def to_canonical(self) -> np.ndarray:
        """Return the 2x3 affine mapping image pixels -> canonical table space."""
        a = max(self.major / 2.0, 1e-6)
        b = max(self.minor / 2.0, 1e-6)
        theta = np.deg2rad(self.angle)

        # Rotate the major axis onto x, squash both axes to CANON_RADIUS, then put
        # the original tilt back so the table's in-image orientation is preserved.
        c, s = np.cos(theta), np.sin(theta)
        rot_to_axis = np.array([[c, s], [-s, c]])
        scale = np.diag([CANON_RADIUS / a, CANON_RADIUS / b])
        rot_back = np.array([[c, -s], [s, c]])
        linear = rot_back @ scale @ rot_to_axis

        offset = np.array([CANON_SIZE / 2.0, CANON_SIZE / 2.0]) - linear @ np.array(
            [self.cx, self.cy]
        )
        return np.hstack([linear, offset.reshape(2, 1)])

    def warp(self, frame: np.ndarray) -> np.ndarray:
        """Warp a frame into canonical table space."""
        return cv2.warpAffine(
            frame,
            self.to_canonical(),
            (CANON_SIZE, CANON_SIZE),
            flags=cv2.INTER_LINEAR,
            borderMode=cv2.BORDER_CONSTANT,
        )

    def points_to_canonical(self, pts: np.ndarray) -> np.ndarray:
        """Map an (N,2) array of image points into canonical table space."""
        pts = np.asarray(pts, dtype=np.float64).reshape(-1, 2)
        m = self.to_canonical()
        return pts @ m[:, :2].T + m[:, 2]

    def points_from_canonical(self, pts: np.ndarray) -> np.ndarray:
        """Map an (N,2) array of canonical points back into image pixels."""
        pts = np.asarray(pts, dtype=np.float64).reshape(-1, 2)
        m = self.to_canonical()
        inv = np.linalg.inv(m[:, :2])
        return (pts - m[:, 2]) @ inv.T


def _cane_mask(frame: np.ndarray) -> np.ndarray:
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    mask = cv2.inRange(hsv, _CANE_LOWER, _CANE_UPPER)
    # The weave is full of holes; close hard enough to make the top a solid blob.
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel, iterations=2)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel, iterations=1)
    return mask


def fit_table(frame: np.ndarray) -> TableFit | None:
    """Fit the table ellipse in a single frame, or None if no plausible table."""
    h, w = frame.shape[:2]
    mask = _cane_mask(frame)

    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None

    frame_area = float(h * w)
    best: TableFit | None = None
    for contour in contours:
        if len(contour) < 5:
            continue
        # Cards and hands sit *on* the table and punch holes in the cane mask, so
        # use the convex hull -- we want the outline of the top, not of the weave.
        hull = cv2.convexHull(contour)
        if cv2.contourArea(hull) < _MIN_AREA_FRAC * frame_area:
            continue
        if len(hull) < 5:
            continue

        (cx, cy), (axis_a, axis_b), angle = cv2.fitEllipse(hull)
        major, minor = max(axis_a, axis_b), min(axis_a, axis_b)
        if minor <= 1.0 or major / minor > _MAX_AXIS_RATIO:
            continue

        ellipse_area = np.pi * (major / 2.0) * (minor / 2.0)
        if not (_MIN_AREA_FRAC * frame_area <= ellipse_area <= _MAX_AREA_FRAC * frame_area):
            continue

        # How much of the fitted ellipse is really cane? Guards against latching
        # onto a wooden bench or a stretch of warm-toned floor.
        probe = np.zeros((h, w), dtype=np.uint8)
        cv2.ellipse(probe, ((cx, cy), (axis_a, axis_b), angle), 255, -1)
        probe_area = float(np.count_nonzero(probe))
        if probe_area < 1.0:
            continue
        score = float(np.count_nonzero(cv2.bitwise_and(probe, mask))) / probe_area
        if score < 0.45:
            continue

        # fitEllipse reports the angle of axis_a; re-express it for the major axis.
        major_angle = angle if axis_a >= axis_b else angle + 90.0
        candidate = TableFit(cx, cy, major, minor, major_angle % 180.0, score)
        if best is None or ellipse_area > best.major * best.minor * np.pi / 4.0:
            best = candidate

    return best


class TableTracker:
    """Temporally smoothed table fit.

    Per-frame fits jitter and occasionally fail outright (a hand sweeping across
    the top, a bad exposure). This keeps an exponential moving average and holds
    the last good fit through short dropouts, so canonical coordinates stay
    continuous across a trick.
    """

    def __init__(self, alpha: float = 0.2, max_hold: int = 90) -> None:
        self.alpha = alpha
        self.max_hold = max_hold
        self._state: TableFit | None = None
        self._misses = 0

    @property
    def current(self) -> TableFit | None:
        return self._state

    def update(self, frame: np.ndarray) -> TableFit | None:
        fit = fit_table(frame)
        if fit is None:
            self._misses += 1
            if self._misses > self.max_hold:
                self._state = None
            return self._state

        self._misses = 0
        if self._state is None:
            self._state = fit
            return self._state

        a = self.alpha
        prev = self._state
        # Average the tilt on the doubled angle: the major axis is 180-periodic, so
        # naive averaging would tear when the fit crosses 0/180.
        prev_r, cur_r = np.deg2rad(prev.angle * 2), np.deg2rad(fit.angle * 2)
        blended = np.arctan2(
            (1 - a) * np.sin(prev_r) + a * np.sin(cur_r),
            (1 - a) * np.cos(prev_r) + a * np.cos(cur_r),
        )
        self._state = TableFit(
            cx=(1 - a) * prev.cx + a * fit.cx,
            cy=(1 - a) * prev.cy + a * fit.cy,
            major=(1 - a) * prev.major + a * fit.major,
            minor=(1 - a) * prev.minor + a * fit.minor,
            angle=float(np.rad2deg(blended) / 2.0) % 180.0,
            score=(1 - a) * prev.score + a * fit.score,
        )
        return self._state

Overwriting registration.py


In [13]:
%%writefile regions.py
"""Region semantics in canonical table space.

A card's *role* in this footage is positional, not visual: the same 3 of diamonds
is a trump indicator on the trump spot and would be meaningless in the trick area.
This module turns a canonical-space position into a role, and -- for trick cards --
into a player seat.

Two distinct mechanisms, on purpose:

* **Role** (trick / trump / counter / hand) is decided by *absolute* position in
  canonical space, since those areas are fixed properties of how the table is laid
  out. They are configurable because the trump and counter spots are conventions of
  this particular group, not of the game.

* **Seat** is decided by the *bearing of a card from the centroid of the current
  trick*, not by absolute position. The four played cards land in a cross, and the
  cross as a whole shifts around the table between tricks. Measuring each card
  relative to its own trick's centre is invariant to that wander, and to any
  residual rotation the registration step leaves behind.
"""

from __future__ import annotations

import json
import math
from dataclasses import dataclass, field
from enum import Enum

import numpy as np

from .registration import CANON_RADIUS, CANON_SIZE

CANON_CENTER = (CANON_SIZE / 2.0, CANON_SIZE / 2.0)


class Role(str, Enum):
    """What a detected card means, given where it sits."""

    TRICK = "trick"  # a played card in the current trick
    TRUMP = "trump"  # the trump indicator (suit matters, rank is noise)
    COUNTER = "counter"  # one of the four 6s used as a point counter
    HAND = "hand"  # a player's face-down pile along the table edge
    UNKNOWN = "unknown"


@dataclass(frozen=True)
class Disc:
    """A circular region in canonical space, as fractions of the table radius."""

    cx: float
    cy: float
    radius: float

    def contains(self, x: float, y: float) -> bool:
        return math.hypot(x - self.cx, y - self.cy) <= self.radius


@dataclass
class TableLayout:
    """Where each role lives in canonical table space.

    Defaults are derived from the canonical persistence maps of both gameplay
    videos: three hand piles hug the top edge, and the trick cross sits in the
    lower-middle of the table. The trump and counter spots are deliberately wide
    and overridable -- they were inferred from a handful of frames, so they are
    config, not fact.
    """

    # Cards beyond this fraction of the table radius, in the upper half, are piles.
    hand_band_radius: float = 0.62
    hand_band_max_angle: float = 115.0  # degrees from "up", either side

    trick_area: Disc = field(
        default_factory=lambda: Disc(
            cx=CANON_CENTER[0] - 0.04 * CANON_RADIUS,
            cy=CANON_CENTER[1] + 0.22 * CANON_RADIUS,
            radius=0.72 * CANON_RADIUS,
        )
    )
    trump_spot: Disc | None = None
    counter_spot: Disc | None = None

    def to_json(self, path: str) -> None:
        def enc(d: Disc | None):
            return None if d is None else {"cx": d.cx, "cy": d.cy, "radius": d.radius}

        payload = {
            "hand_band_radius": self.hand_band_radius,
            "hand_band_max_angle": self.hand_band_max_angle,
            "trick_area": enc(self.trick_area),
            "trump_spot": enc(self.trump_spot),
            "counter_spot": enc(self.counter_spot),
        }
        with open(path, "w") as fh:
            json.dump(payload, fh, indent=2)

    @classmethod
    def from_json(cls, path: str) -> "TableLayout":
        with open(path) as fh:
            payload = json.load(fh)

        def dec(d):
            return None if d is None else Disc(d["cx"], d["cy"], d["radius"])

        return cls(
            hand_band_radius=payload["hand_band_radius"],
            hand_band_max_angle=payload["hand_band_max_angle"],
            trick_area=dec(payload["trick_area"]),
            trump_spot=dec(payload["trump_spot"]),
            counter_spot=dec(payload["counter_spot"]),
        )

    def classify(self, x: float, y: float) -> Role:
        """Assign a role to a canonical-space point (centre of a detected card)."""
        # Explicitly configured spots win: they are narrow and deliberate, whereas
        # the hand band and trick area are broad catch-alls that would swallow them.
        if self.trump_spot is not None and self.trump_spot.contains(x, y):
            return Role.TRUMP
        if self.counter_spot is not None and self.counter_spot.contains(x, y):
            return Role.COUNTER

        dx, dy = x - CANON_CENTER[0], y - CANON_CENTER[1]
        radius_frac = math.hypot(dx, dy) / CANON_RADIUS
        # Bearing measured from "up" (negative y), positive clockwise.
        bearing = math.degrees(math.atan2(dx, -dy))
        if radius_frac >= self.hand_band_radius and abs(bearing) <= self.hand_band_max_angle:
            return Role.HAND

        if self.trick_area.contains(x, y):
            return Role.TRICK
        if radius_frac > 1.05:
            return Role.UNKNOWN
        return Role.TRICK


# Seat order around the cross, clockwise starting from the arm nearest the camera.
# The fourth player sits off-frame on the camera side, so their card lands in the
# south arm; the three visible players fill west, north and east.
_SEAT_BEARINGS = {
    "south": 180.0,
    "west": 270.0,
    "north": 0.0,
    "east": 90.0,
}


def assign_seats(points: np.ndarray) -> list[str]:
    """Map trick-card positions to cross arms by bearing from the trick centroid.

    ``points`` is an (N,2) array of canonical-space card centres belonging to one
    trick (N <= 4). Returns the arm name per point. Each arm is used at most once:
    with fewer than four cards the centroid is pulled off-centre, so a greedy
    nearest-bearing assignment would happily put two cards on the same arm.
    """
    pts = np.asarray(points, dtype=np.float64).reshape(-1, 2)
    if len(pts) == 0:
        return []

    centroid = pts.mean(axis=0)
    bearings = np.degrees(np.arctan2(pts[:, 0] - centroid[0], -(pts[:, 1] - centroid[1])))

    names = list(_SEAT_BEARINGS)
    # Cost = angular distance from each card to each arm, then a small exhaustive
    # search for the lowest-cost one-to-one assignment (at most 4! = 24 options).
    cost = np.zeros((len(pts), len(names)))
    for i, b in enumerate(bearings):
        for j, name in enumerate(names):
            diff = abs((b - _SEAT_BEARINGS[name] + 180.0) % 360.0 - 180.0)
            cost[i, j] = diff

    from itertools import permutations

    best_total, best_choice = float("inf"), None
    for choice in permutations(range(len(names)), len(pts)):
        total = sum(cost[i, j] for i, j in enumerate(choice))
        if total < best_total:
            best_total, best_choice = total, choice

    assert best_choice is not None
    return [names[j] for j in best_choice]

Overwriting regions.py


In [14]:
%%writefile tracking.py
"""Temporal layer: frames -> tricks -> games -> {player: card} per trick.

Single-frame detection is not the deliverable. What matters is, per game and per
trick, which card each player played. Getting there needs three things that only
exist across time:

1. **Trick segmentation.** Cards accumulate in the play area, then get swept up
   when the trick is taken. That sweep is the boundary. A trick is the span
   between two sweeps.

2. **Voting within a trick.** A card sits on the table for many frames, so it gets
   classified many times. The consensus over those frames is far stronger than any
   single frame -- which is the point, given a played card is only ~55x85 px.

3. **The uniqueness constraint.** Each of the 32 cards is played exactly once per
   game. That turns identification into a global assignment problem rather than 32
   independent guesses, and it repairs errors that no amount of per-frame
   confidence would: if the model likes 9H for two different slots, at most one can
   keep it, and the other is forced to its best remaining option.

Seating is only read off frames where the trick is at its fullest. With one or two
cards down, the centroid sits on top of the cards themselves and the bearing that
:func:`assign_seats` relies on is meaningless.
"""

from __future__ import annotations

from collections import defaultdict
from dataclasses import dataclass, field

import numpy as np

from .regions import assign_seats

# Seats, in the fixed cross arrangement, mapped to the engine's player numbers.
# Player 4 sits off-frame nearest the camera and plays into the south arm.
SEAT_TO_PLAYER: dict[str, int] = {"south": 4, "west": 1, "north": 2, "east": 3}


@dataclass(frozen=True)
class Detection:
    """One detected trick card in canonical table space."""

    code: str
    conf: float
    x: float
    y: float


@dataclass
class RawTrick:
    """A segmented trick, before uniqueness is enforced across the game."""

    start_frame: int
    end_frame: int
    peak_cards: int
    # seat -> {card code -> accumulated confidence}
    votes: dict[str, dict[str, float]] = field(default_factory=dict)

    def best_by_seat(self) -> dict[str, str]:
        """Greedy per-seat winner, ignoring the uniqueness constraint."""
        out = {}
        for seat, tally in self.votes.items():
            if tally:
                out[seat] = max(tally, key=tally.get)
        return out


class TrickSegmenter:
    """Turn a stream of per-frame detections into discrete tricks.

    ``min_empty_frames`` is how many consecutive card-free observations count as a
    sweep; it debounces the frequent single-frame dropout (a hand crossing the
    table, a bad exposure) that would otherwise split one trick into several.
    """

    def __init__(
        self,
        min_empty_frames: int = 3,
        min_peak_cards: int = 2,
        max_cards: int = 4,
        min_trick_frames: int = 0,
    ) -> None:
        self.min_empty_frames = min_empty_frames
        self.min_peak_cards = min_peak_cards
        self.max_cards = max_cards
        self.min_trick_frames = min_trick_frames
        # Segments dropped for being too short/weak, for diagnostics.
        self.rejected = 0
        self._observations: list[tuple[int, list[Detection]]] = []
        self._empty_streak = 0
        self._start_frame: int | None = None
        self.tricks: list[RawTrick] = []

    def feed(self, frame_index: int, detections: list[Detection]) -> None:
        detections = sorted(detections, key=lambda d: -d.conf)[: self.max_cards]

        if detections:
            self._empty_streak = 0
            if self._start_frame is None:
                self._start_frame = frame_index
            self._observations.append((frame_index, detections))
            return

        self._empty_streak += 1
        if self._observations and self._empty_streak >= self.min_empty_frames:
            self._close(frame_index)

    def finish(self) -> list[RawTrick]:
        """Close any trick still open at end of stream and return all tricks."""
        if self._observations:
            self._close(self._observations[-1][0])
        return self.tricks

    def _close(self, end_frame: int) -> None:
        observations = self._observations
        self._observations = []
        self._empty_streak = 0
        start = self._start_frame
        self._start_frame = None

        peak = max(len(d) for _, d in observations)
        if peak < self.min_peak_cards or start is None:
            self.rejected += 1
            return  # stray detections, not a real trick

        # A real trick lasts as long as it takes four players to play. Anything
        # briefer is detector flicker: on the real footage a naive emptiness test
        # fires an order of magnitude more often than there are actual tricks.
        if observations[-1][0] - start < self.min_trick_frames:
            self.rejected += 1
            return

        # Seat only from the fullest frames -- see the module docstring.
        fullest = [(idx, dets) for idx, dets in observations if len(dets) == peak]
        votes: dict[str, dict[str, float]] = defaultdict(lambda: defaultdict(float))
        for _, dets in fullest:
            seats = assign_seats(np.array([[d.x, d.y] for d in dets]))
            for det, seat in zip(dets, seats):
                votes[seat][det.code] += det.conf

        self.tricks.append(
            RawTrick(
                start_frame=start,
                end_frame=end_frame,
                peak_cards=peak,
                votes={s: dict(t) for s, t in votes.items()},
            )
        )


def split_games(
    tricks: list[RawTrick],
    tricks_per_game: int = 8,
    deal_gap_frames: int | None = None,
) -> list[list[RawTrick]]:
    """Group tricks into games.

    A game is ``tricks_per_game`` tricks. When ``deal_gap_frames`` is given, an
    unusually long card-free stretch (the deal) also closes a game, which recovers
    the grouping when a trick is missed or spuriously split.
    """
    games: list[list[RawTrick]] = []
    current: list[RawTrick] = []

    for i, trick in enumerate(tricks):
        if current and deal_gap_frames is not None:
            gap = trick.start_frame - current[-1].end_frame
            if gap >= deal_gap_frames:
                games.append(current)
                current = []
        current.append(trick)
        if len(current) == tricks_per_game:
            games.append(current)
            current = []

    if current:
        games.append(current)
    return games


def enforce_uniqueness(
    game_tricks: list[RawTrick], all_codes: list[str]
) -> list[dict[str, str]]:
    """Resolve a whole game at once, so each card is used at most once.

    Returns one ``{seat: card_code}`` mapping per trick. Solves a maximum-weight
    assignment over every (trick, seat) slot against the 32 cards, using the voted
    confidences as weights -- rather than taking each slot's argmax independently
    and hoping no card gets claimed twice.
    """
    slots: list[tuple[int, str]] = []
    for t_idx, trick in enumerate(game_tricks):
        for seat in trick.votes:
            slots.append((t_idx, seat))

    if not slots:
        return [{} for _ in game_tricks]

    code_index = {c: i for i, c in enumerate(all_codes)}
    score = np.zeros((len(slots), len(all_codes)))
    for row, (t_idx, seat) in enumerate(slots):
        for codeval, weight in game_tricks[t_idx].votes[seat].items():
            if codeval in code_index:
                score[row, code_index[codeval]] = weight

    try:
        from scipy.optimize import linear_sum_assignment

        rows, cols = linear_sum_assignment(score, maximize=True)
        pairs = list(zip(rows, cols))
    except ImportError:  # pragma: no cover - scipy ships with Colab
        pairs = _greedy_assignment(score)

    result: list[dict[str, str]] = [{} for _ in game_tricks]
    for row, col in pairs:
        # A slot with no vote mass at all would otherwise be handed an arbitrary
        # leftover card purely to complete the matching.
        if score[row, col] <= 0:
            continue
        t_idx, seat = slots[row]
        result[t_idx][seat] = all_codes[col]
    return result


def _greedy_assignment(score: np.ndarray) -> list[tuple[int, int]]:
    """Fallback for the assignment problem: repeatedly take the best free pair."""
    pairs = []
    used_rows: set[int] = set()
    used_cols: set[int] = set()
    order = np.dstack(np.unravel_index(np.argsort(score, axis=None)[::-1], score.shape))[0]
    for row, col in order:
        if row in used_rows or col in used_cols:
            continue
        used_rows.add(int(row))
        used_cols.add(int(col))
        pairs.append((int(row), int(col)))
    return pairs


def to_player_mapping(seat_mapping: dict[str, str]) -> dict[int, str]:
    """Convert ``{seat: code}`` to ``{player_number: code}``."""
    return {SEAT_TO_PLAYER[seat]: codeval for seat, codeval in seat_mapping.items()}

Overwriting tracking.py


In [15]:
%%writefile card_faces.py
"""Procedurally rendered card faces for the 32-card *29* deck.

These exist so the pipeline is trainable *before* real card scans are available.
Faces are drawn geometrically rather than with a font, so nothing depends on
which glyphs the runtime happens to ship.

They are a stand-in, not a substitute: a classifier trained purely on these will
learn this synthetic look. Drop real scans into ``data/reference/<CODE>.png``
(e.g. ``JS.png``, ``10H.png``) and ``load_gallery`` prefers them automatically.
"""

from __future__ import annotations

import os

import numpy as np
from PIL import Image, ImageDraw

CARD_W, CARD_H = 140, 200
RED = (200, 30, 40)
BLACK = (25, 25, 30)

RANKS = ["7", "8", "9", "10", "J", "Q", "K", "A"]
SUITS = ["S", "H", "D", "C"]
SUIT_COLOR = {"S": BLACK, "C": BLACK, "H": RED, "D": RED}


def _diamond(draw, cx, cy, r, color):
    draw.polygon([(cx, cy - r), (cx + r * 0.72, cy), (cx, cy + r), (cx - r * 0.72, cy)], fill=color)


def _heart(draw, cx, cy, r, color):
    draw.ellipse([cx - r, cy - r * 0.95, cx, cy + r * 0.1], fill=color)
    draw.ellipse([cx, cy - r * 0.95, cx + r, cy + r * 0.1], fill=color)
    draw.polygon([(cx - r * 0.97, cy - r * 0.1), (cx + r * 0.97, cy - r * 0.1), (cx, cy + r)], fill=color)


def _spade(draw, cx, cy, r, color):
    draw.polygon([(cx, cy - r), (cx + r * 0.95, cy + r * 0.25), (cx - r * 0.95, cy + r * 0.25)], fill=color)
    draw.ellipse([cx - r, cy - r * 0.1, cx, cy + r * 0.7], fill=color)
    draw.ellipse([cx, cy - r * 0.1, cx + r, cy + r * 0.7], fill=color)
    draw.polygon([(cx - r * 0.3, cy + r), (cx + r * 0.3, cy + r), (cx, cy + r * 0.25)], fill=color)


def _club(draw, cx, cy, r, color):
    draw.ellipse([cx - r * 0.42, cy - r, cx + r * 0.42, cy - r * 0.16], fill=color)
    draw.ellipse([cx - r, cy - r * 0.3, cx - r * 0.16, cy + r * 0.54], fill=color)
    draw.ellipse([cx + r * 0.16, cy - r * 0.3, cx + r, cy + r * 0.54], fill=color)
    draw.polygon([(cx - r * 0.32, cy + r), (cx + r * 0.32, cy + r), (cx, cy + r * 0.2)], fill=color)


_PIP = {"S": _spade, "H": _heart, "D": _diamond, "C": _club}


def _draw_glyph(draw, suit, cx, cy, r, color):
    _PIP[suit](draw, cx, cy, r, color)


# Seven-segment style digits/letters, drawn as strokes so no font is required.
_SEGMENTS = {
    "7": [(0, 0, 1, 0), (1, 0, 1, 1), (1, 1, 1, 2)],
    "8": [(0, 0, 1, 0), (0, 0, 0, 1), (1, 0, 1, 1), (0, 1, 1, 1), (0, 1, 0, 2), (1, 1, 1, 2), (0, 2, 1, 2)],
    "9": [(0, 0, 1, 0), (0, 0, 0, 1), (1, 0, 1, 1), (0, 1, 1, 1), (1, 1, 1, 2), (0, 2, 1, 2)],
    "1": [(1, 0, 1, 1), (1, 1, 1, 2)],
    "0": [(0, 0, 1, 0), (0, 0, 0, 1), (1, 0, 1, 1), (0, 1, 0, 2), (1, 1, 1, 2), (0, 2, 1, 2)],
    "J": [(1, 0, 1, 1), (1, 1, 1, 2), (0, 2, 1, 2), (0, 1, 0, 2)],
    "Q": [(0, 0, 1, 0), (0, 0, 0, 1), (1, 0, 1, 1), (0, 1, 0, 2), (1, 1, 1, 2), (0, 2, 1, 2)],
    "K": [(0, 0, 0, 1), (0, 1, 0, 2), (0, 1, 1, 1), (1, 0, 0, 1), (0, 1, 1, 2)],
    "A": [(0, 0, 1, 0), (0, 0, 0, 1), (1, 0, 1, 1), (0, 1, 1, 1), (0, 1, 0, 2), (1, 1, 1, 2)],
}


def _draw_text(draw, text, x, y, w, h, color, width=3):
    """Render a rank string as segment strokes inside a (w,h) box at (x,y)."""
    per = w / (len(text) * 1.35)
    for i, ch in enumerate(text):
        ox = x + i * per * 1.35
        for (x0, y0, x1, y1) in _SEGMENTS.get(ch, []):
            draw.line(
                [(ox + x0 * per, y + y0 * h / 2), (ox + x1 * per, y + y1 * h / 2)],
                fill=color,
                width=width,
            )


# Pip layouts as (x, y) in unit card space, for the numeric ranks.
_LAYOUTS = {
    "7": [(.5, .18), (.28, .3), (.72, .3), (.28, .5), (.72, .5), (.28, .72), (.72, .72)],
    "8": [(.28, .2), (.72, .2), (.28, .4), (.72, .4), (.28, .6), (.72, .6), (.28, .8), (.72, .8)],
    "9": [(.28, .2), (.72, .2), (.28, .4), (.72, .4), (.5, .5), (.28, .6), (.72, .6), (.28, .8), (.72, .8)],
    "10": [(.28, .18), (.72, .18), (.28, .35), (.72, .35), (.5, .27), (.28, .65), (.72, .65), (.5, .73), (.28, .82), (.72, .82)],
    "A": [(.5, .5)],
}


def render_card(rank: str, suit: str) -> np.ndarray:
    """Render one card face as an RGB numpy array."""
    img = Image.new("RGB", (CARD_W, CARD_H), (250, 249, 245))
    draw = ImageDraw.Draw(img)
    draw.rounded_rectangle([1, 1, CARD_W - 2, CARD_H - 2], radius=10, outline=(180, 180, 180), width=2)
    color = SUIT_COLOR[suit]

    # Corner indices, top-left and (rotated) bottom-right.
    for flip in (False, True):
        if flip:
            corner = Image.new("RGB", (34, 52), (250, 249, 245))
            cdraw = ImageDraw.Draw(corner)
            _draw_text(cdraw, rank, 3, 3, 24, 24, color, width=3)
            _draw_glyph(cdraw, suit, 17, 41, 8, color)
            img.paste(corner.rotate(180), (CARD_W - 38, CARD_H - 56))
        else:
            _draw_text(draw, rank, 7, 7, 24, 24, color, width=3)
            _draw_glyph(draw, suit, 21, 45, 8, color)

    if rank in _LAYOUTS:
        for (ux, uy) in _LAYOUTS[rank]:
            r = 20 if rank == "A" else 11
            _draw_glyph(draw, suit, ux * CARD_W, uy * CARD_H, r, color)
    else:
        # Court cards: a framed panel with a large central glyph.
        draw.rectangle([38, 42, CARD_W - 38, CARD_H - 42], outline=color, width=3)
        _draw_glyph(draw, suit, CARD_W / 2, CARD_H / 2, 26, color)
        _draw_text(draw, rank, CARD_W / 2 - 12, CARD_H / 2 - 62, 24, 24, color, width=3)

    return np.array(img)


def load_gallery(reference_dir: str = "data/reference") -> dict[str, np.ndarray]:
    """Return ``{card_code: RGB array}``, preferring real scans when present."""
    gallery: dict[str, np.ndarray] = {}
    real = 0
    for suit in SUITS:
        for rank in RANKS:
            codeval = f"{rank}{suit}"
            path = os.path.join(reference_dir, f"{codeval}.png")
            if os.path.exists(path):
                img = Image.open(path).convert("RGB").resize((CARD_W, CARD_H))
                gallery[codeval] = np.array(img)
                real += 1
            else:
                gallery[codeval] = render_card(rank, suit)
    print(f"gallery: {len(gallery)} cards ({real} real scans, {len(gallery) - real} procedural)")
    return gallery

Overwriting card_faces.py


In [16]:
%%writefile synthetic_canon.py
"""Compose synthetic training scenes directly in canonical table space.

Training and inference must share a geometry. Inference warps each real frame
onto the canonical table disc, so scenes are composited there too -- rather than
in raw frame space, which would force the detector to also absorb the hand-held
camera's drift and foreshortening.

Backgrounds are real empty-table crops harvested from the gameplay videos, so
the model sees the actual cane weave and lighting rather than a flat colour.
"""

from __future__ import annotations

import random

import cv2
import numpy as np

from p29.vision.registration import CANON_RADIUS, CANON_SIZE
from p29.vision.regions import CANON_CENTER

CARD_SCALE = 0.34  # card height as a fraction of the table radius


def _place(scene, card_rgb, cx, cy, angle, scale, brightness):
    """Alpha-composite a rotated card onto the scene; return its bbox."""
    h, w = card_rgb.shape[:2]
    target_h = max(8, int(CANON_RADIUS * CARD_SCALE * scale))
    target_w = max(6, int(target_h * w / h))
    card = cv2.resize(card_rgb, (target_w, target_h), interpolation=cv2.INTER_AREA)
    card = np.clip(card.astype(np.float32) * brightness, 0, 255).astype(np.uint8)

    mask = np.full((target_h, target_w), 255, dtype=np.uint8)
    diag = int(np.hypot(target_w, target_h)) + 4
    pad_card = np.zeros((diag, diag, 3), dtype=np.uint8)
    pad_mask = np.zeros((diag, diag), dtype=np.uint8)
    y0, x0 = (diag - target_h) // 2, (diag - target_w) // 2
    pad_card[y0:y0 + target_h, x0:x0 + target_w] = card
    pad_mask[y0:y0 + target_h, x0:x0 + target_w] = mask

    rot = cv2.getRotationMatrix2D((diag / 2, diag / 2), angle, 1.0)
    pad_card = cv2.warpAffine(pad_card, rot, (diag, diag))
    pad_mask = cv2.warpAffine(pad_mask, rot, (diag, diag))

    top, left = int(cy - diag / 2), int(cx - diag / 2)
    sy0, sx0 = max(0, top), max(0, left)
    sy1, sx1 = min(CANON_SIZE, top + diag), min(CANON_SIZE, left + diag)
    if sy1 <= sy0 or sx1 <= sx0:
        return None

    cy0, cx0 = sy0 - top, sx0 - left
    sub_card = pad_card[cy0:cy0 + (sy1 - sy0), cx0:cx0 + (sx1 - sx0)]
    sub_mask = pad_mask[cy0:cy0 + (sy1 - sy0), cx0:cx0 + (sx1 - sx0)]
    alpha = (sub_mask.astype(np.float32) / 255.0)[..., None]
    region = scene[sy0:sy1, sx0:sx1].astype(np.float32)
    scene[sy0:sy1, sx0:sx1] = (region * (1 - alpha) + sub_card.astype(np.float32) * alpha).astype(np.uint8)

    ys, xs = np.nonzero(sub_mask)
    if len(xs) == 0:
        return None
    return (sx0 + xs.min(), sy0 + ys.min(), sx0 + xs.max(), sy0 + ys.max())


def make_scene(gallery, backgrounds, rng: random.Random):
    """Build one canonical-space scene. Returns (image, [(code, bbox), ...])."""
    bg = backgrounds[rng.randrange(len(backgrounds))].copy()
    scene = cv2.resize(bg, (CANON_SIZE, CANON_SIZE))

    codes = list(gallery)
    rng.shuffle(codes)
    n_cards = rng.randint(1, 4)
    chosen = codes[:n_cards]

    # Lay the trick out as a cross, jittered, the way it actually falls on table.
    centre = (
        CANON_CENTER[0] + rng.uniform(-0.18, 0.18) * CANON_RADIUS,
        CANON_CENTER[1] + rng.uniform(0.05, 0.35) * CANON_RADIUS,
    )
    arm = rng.uniform(0.28, 0.42) * CANON_RADIUS
    slots = [(0, -arm), (arm, 0), (0, arm), (-arm, 0)]
    rng.shuffle(slots)

    global_light = rng.uniform(0.72, 1.18)
    annotations = []
    for codeval, (dx, dy) in zip(chosen, slots):
        cx = centre[0] + dx + rng.uniform(-14, 14)
        cy = centre[1] + dy + rng.uniform(-14, 14)
        box = _place(
            scene,
            gallery[codeval],
            cx,
            cy,
            angle=rng.uniform(-180, 180),
            scale=rng.uniform(0.85, 1.15),
            brightness=global_light * rng.uniform(0.92, 1.08),
        )
        if box is not None:
            annotations.append((codeval, box))

    if rng.random() < 0.5:
        k = rng.choice([3, 5])
        scene = cv2.GaussianBlur(scene, (k, k), 0)
    if rng.random() < 0.3:
        noise = rng.uniform(3, 11)
        scene = np.clip(scene.astype(np.float32) + np.random.normal(0, noise, scene.shape), 0, 255).astype(np.uint8)

    return scene, annotations


def _table_disc_mask():
    """The whole table top, not just the play area.

    Emptiness is scored over the entire table on purpose. Any real face-up card
    left in a background is an *unlabelled* card in a training image, which teaches
    the detector to ignore exactly what it is meant to find -- and that applies
    wherever on the table it sits, not only in the middle. The face-down hand piles
    are always present and so contribute a near-constant offset that ranking
    absorbs harmlessly.
    """
    yy, xx = np.mgrid[0:CANON_SIZE, 0:CANON_SIZE]
    return np.hypot(xx - CANON_CENTER[0], yy - CANON_CENTER[1]) < 0.98 * CANON_RADIUS


def harvest_backgrounds(video_paths, tracker_cls, max_per_video=60, sample_every=2.0):
    """Collect canonical table crops whose play area is as empty as possible.

    Emptiness is *ranked*, not thresholded. The cane weave's own highlights read as
    bright and desaturated, so even a bare table scores ~8% "card-like" pixels in
    the table -- any absolute cutoff either takes everything or nothing. Taking
    the lowest-scoring frames per video needs no tuned constant and adapts to
    whatever the lighting happens to be.

    Two passes, so only the selected frames are ever held in memory: score first,
    then re-read the winners using the table fit recorded alongside each score.
    """
    play = _table_disc_mask()
    backgrounds = []

    for path in video_paths:
        cap = cv2.VideoCapture(path)
        fps = max(cap.get(cv2.CAP_PROP_FPS), 1.0)
        step = max(1, int(fps * sample_every))

        tracker = tracker_cls()
        scored = []
        i = -1
        while True:  # sequential decode; seeking per sample is far slower
            if not cap.grab():
                break
            i += 1
            if i % step:
                continue
            ok, frame = cap.retrieve()
            if not ok:
                break
            fit = tracker.update(frame)
            if fit is None:
                continue
            hsv = cv2.cvtColor(fit.warp(frame), cv2.COLOR_BGR2HSV)
            cardish = (hsv[:, :, 2] > 150) & (hsv[:, :, 1] < 60)
            scored.append((float((cardish & play).sum() / play.sum()), i, fit))

        # Only the winners are re-read, so peak memory stays at max_per_video frames.
        scored.sort(key=lambda t: t[0])
        for _, i, fit in scored[:max_per_video]:
            cap.set(cv2.CAP_PROP_POS_FRAMES, i)
            ok, frame = cap.read()
            if ok:
                backgrounds.append(fit.warp(frame))
        cap.release()

    return backgrounds

Overwriting synthetic_canon.py
